In [5]:
import requests
from bs4 import BeautifulSoup
import re

# List of boxer names to scrape (you can add more names or load from a file)
boxers = [
    "Floyd Mayweather Jr.", "Manny Pacquiao", "Mike Tyson", "Tyson Fury", 
    "Canelo Álvarez", "Anthony Joshua", "Deontay Wilder", 
    "Oleksandr Usyk", "Vasiliy Lomachenko", "Gennady Golovkin",
]

# Function to scrape one boxer's data from Wikipedia
def scrape_boxer_info(name):
    # Construct Wikipedia URL (replace spaces with underscores)
    url = "https://en.wikipedia.org/wiki/" + name.replace(' ', '_')
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    if resp.status_code != 200:
        print(f"Failed to retrieve page for {name} (status {resp.status_code})")
        return None
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    # Find the infobox table in the page
    infobox = soup.find("table", {"class": "infobox"})
    if infobox is None:
        print(f"No infobox found for {name} – skipping.")
        return None
    
    data = {"Name": name}
    # Go through each table row in the infobox
    for row in infobox.find_all("tr"):
        header = row.find("th")
        value = row.find("td")
        if not header or not value:
            continue  # skip rows that are not "header: value" pairs (e.g. section headers)
        field = header.get_text(strip=True)
        val_text = value.get_text(" ", strip=True)  # get text inside td
        
        # Extract relevant fields
        if field.startswith("Weight"):  # "Weight(s)"
            # If multiple weight classes are listed as a <ul> list, join them
            # Check for list items in the value
            weight_classes = [li.get_text(strip=True) for li in value.find_all("li")]
            if weight_classes:
                data["Weight"] = ", ".join(weight_classes)
            else:
                data["Weight"] = val_text
        elif field == "Height":
            data["Height"] = val_text
        elif field == "Reach":
            data["Reach"] = val_text
        elif field == "Stance" or field == "Style":
            # Wikipedia uses 'Stance' for boxing (Style is used in some martial arts infoboxes)
            data["Stance"] = val_text
        elif field == "Total fights":
            data["Total_fights"] = val_text
        elif field == "Wins":
            data["Wins"] = val_text
        elif field == "Losses":
            data["Losses"] = val_text
        elif field == "Draws":
            data["Draws"] = val_text
        elif field == "No contests":  # if present
            data["No_contests"] = val_text
        elif field == "Born":
            # Extract age from the Born field if available
            # Born field often contains birthdate and age like "January 17, 1978 (age 45)"
            age_match = re.search(r'\(age\s+(\d+)\)', val_text)
            if age_match:
                data["Age"] = age_match.group(1)
            else:
                data["Age"] = None  # age not found (maybe the person is deceased or info not available)
        # (We skip other fields like nationality, etc., not requested)
    
    # If age wasn't in Born (e.g., deceased boxers won't have an age there), check for Died field
    if "Age" not in data:
        died_field = infobox.find("th", string="Died")
        if died_field:
            died_text = died_field.find_next("td").get_text(" ", strip=True)
            age_match = re.search(r'\(aged\s+(\d+)\)', died_text)
            if age_match:
                data["Age"] = age_match.group(1)  # age at death
    
    return data



### remember you like doing these things, do not compare yourself to others and no matter what you do, its your diploma and no one knows themselves more
### than you do so stop worrying about everybody else

# Scrape data for all boxers in the list
dataset = []
for name in boxers:
    info = scrape_boxer_info(name)
    if info:
        dataset.append(info)

# Display the collected data (for example, print each boxer's info)
for boxer in dataset:
    print(boxer)
    
    

{'Name': 'Floyd Mayweather Jr.', 'Age': '48', 'Weight': 'Super featherweight, Lightweight, Light welterweight, Welterweight, Light middleweight', 'Height': '5\xa0ft 8\xa0in (173\xa0cm) [ 1 ]', 'Reach': '72\xa0in (183\xa0cm) [ 1 ]', 'Stance': 'Orthodox', 'Total_fights': '50', 'Wins': '50'}
{'Name': 'Manny Pacquiao', 'Age': '46', 'Weight': 'Light flyweight, Flyweight, Super bantamweight, Featherweight, Super featherweight, Lightweight, Light welterweight, Welterweight, Light middleweight', 'Height': '5\xa0ft 5\xa0in (165\xa0cm) [ 4 ]', 'Reach': '67\xa0in (170\xa0cm) [ 4 ]', 'Stance': 'Southpaw', 'Total_fights': '72', 'Wins': '62', 'Losses': '8', 'Draws': '2'}
{'Name': 'Mike Tyson', 'Age': '59', 'Weight': 'Heavyweight', 'Height': '5\xa0ft 11\xa0in (180\xa0cm) [ 1 ] [ 2 ] [ nb 1 ]', 'Reach': '71\xa0in (180\xa0cm) [ 3 ]', 'Stance': 'Orthodox', 'Total_fights': '59', 'Wins': '50', 'Losses': '7', 'No_contests': '2'}
{'Name': 'Tyson Fury', 'Age': '36', 'Weight': 'Heavyweight', 'Height': '6\xa0f

In [6]:
import pandas as pd

df = pd.DataFrame(dataset)
df.to_csv("boxers_dataset.csv", index=False)
print(df.head())  # print first few rows of the dataframe as a sanity check

                   Name Age  \
0  Floyd Mayweather Jr.  48   
1        Manny Pacquiao  46   
2            Mike Tyson  59   
3            Tyson Fury  36   
4        Canelo Álvarez  34   

                                              Weight  \
0  Super featherweight, Lightweight, Light welter...   
1  Light flyweight, Flyweight, Super bantamweight...   
2                                        Heavyweight   
3                                        Heavyweight   
4  Welterweight, Light middleweight, Middleweight...   

                                     Height                         Reach  \
0                  5 ft 8 in (173 cm) [ 1 ]          72 in (183 cm) [ 1 ]   
1                  5 ft 5 in (165 cm) [ 4 ]          67 in (170 cm) [ 4 ]   
2  5 ft 11 in (180 cm) [ 1 ] [ 2 ] [ nb 1 ]          71 in (180 cm) [ 3 ]   
3                  6 ft 9 in (206 cm) [ 1 ]          85 in (216 cm) [ 1 ]   
4          5 ft 7 + 1 ⁄ 2 in (171 cm) [ 1 ]  70 + 1 ⁄ 2 in (179 cm) [ 1 ]   

           St

In [6]:
df.to_csv("C:/Users/eleon/Downloads/boxers_dataset.csv", index=False)

In [7]:
more_boxers= ["Oleksandr Usyk","Tyson Fury","Daniel Dubois","Joseph Parker","Agit Kabayel","Fabio Wardley","Martin Bakole","Derek Chisora","Michael Hunter",
              
"Frank Sanchez","Moses Itauma","Jared Anderson","Kubrat Pulev","Jarrell Miller","Dillian Whyte","Andy Ruiz Jr.","Murat Gassiev","Justis Huni","Joe Joyce",
"Jai Opetaia","Gilberto Ramirez","Chris Billam‑Smith","Badou Jack","Norair Mikaeljan","Evgeny Romanov","Leon Harth","Andrew Tabiti","Kevin Lerena","Dmitry Bivol",        
"Artur Beterbiev","David Benavidez","David Morrell Jr.","Callum Smith","Joshua Buatsi","Efe Ajagba","Canelo Álvarez","Christian Mbilli","Diego Pacheco","Edgar Berlanga",
"Jermall Charlo","Caleb Plant","Janibek Alimkhanuly","Carlos Adames","Hamzah Sheeraz","Erislandy Lara","Chris Eubank Jr.","Terence Crawford","Vergil Ortiz Jr.",
"Sebastian Fundora","Bakhram Murtazaliev","Israil Madrimov","Tim Tszyu","Jaron Ennis","Brian Norman Jr.","Mario Barrios","Eimantas Stanionis","Devin Haney",
"Teofimo Lopez","Richardson Hitchins","Alberto Puello","Arnold Barboza Jr.","Gary Antuanne Russell","Liam Paro","Subriel Matías","Sandor Martín","Gervonta Davis",
"Shakur Stevenson","Keyshawn Davis","William Zepeda","Lamont Roach","Andy Cruz","Emanuel Navarrete","Anthony Cacace","O’Shaiquie Foster","Jesse Rodriguez",
"Fernando Martinez","Kazuto Ioka","Phumelele Cafu","Carlos Cuadras","Kosei Tanaka","Naoya Inoue","Marlon Tapales","Murodjon Akhmadaliev","Sam Goodman","Luis Nery",
"Alan Picasso Romero","Junto Nakatani","Ryosuke Nishida","Kenshiro Teraji","Seigo Yuri Akui","Francisco Rodriguez Jr.","Ricardo Sandoval","Masamichi Yabuki",
"Rene Santiago","Jonathan Gonzalez","Elwin Soto","Petchmanee CP Freshmart","Shokichi Iwata","Oscar Collazo","Pedro Taduran"]


In [8]:
dataset = []
for name in more_boxers:
    info = scrape_boxer_info(name)
    if info:
        dataset.append(info)

# Display the collected data (for example, print each boxer's info)
for boxer in dataset:
    print(boxer)

No infobox found for Michael Hunter – skipping.
No infobox found for Frank Sanchez – skipping.
No infobox found for Jared Anderson – skipping.
No infobox found for Joe Joyce – skipping.
Failed to retrieve page for Chris Billam‑Smith (status 404)
No infobox found for Evgeny Romanov – skipping.
Failed to retrieve page for Leon Harth (status 404)
Failed to retrieve page for David Morrell Jr. (status 404)
No infobox found for Diego Pacheco – skipping.
Failed to retrieve page for Lamont Roach (status 404)
Failed to retrieve page for O’Shaiquie Foster (status 404)
No infobox found for Jesse Rodriguez – skipping.
No infobox found for Fernando Martinez – skipping.
No infobox found for Luis Nery – skipping.
Failed to retrieve page for Francisco Rodriguez Jr. (status 404)
Failed to retrieve page for Ricardo Sandoval (status 404)
Failed to retrieve page for Rene Santiago (status 404)
No infobox found for Jonathan Gonzalez – skipping.
Failed to retrieve page for Petchmanee CP Freshmart (status 404

In [10]:
import pandas as pd

df = pd.DataFrame(dataset)
df.to_csv("boxers_dataset.csv", index=False)
print(df.head())  # print first few rows of the dataframe as a sanity check


df.to_csv("C:/Users/eleon/Downloads/boxers_dataset.csv", index=False)

             Name Age                      Weight                    Height  \
0  Oleksandr Usyk  38  Cruiserweight, Heavyweight  1.91 m (6 ft 3 in) [ 1 ]   
1      Tyson Fury  36                 Heavyweight  6 ft 9 in (206 cm) [ 1 ]   
2   Daniel Dubois  27                 Heavyweight  6 ft 5 in (196 cm) [ 1 ]   
3   Joseph Parker  33                 Heavyweight  6 ft 4 in (193 cm) [ 1 ]   
4    Agit Kabayel  32                 Heavyweight        6 ft 3 in (191 cm)   

                  Reach          Stance Total_fights Wins Losses Draws  \
0  198 cm (78 in) [ 1 ]        Southpaw           23   23    NaN   NaN   
1  85 in (216 cm) [ 1 ]  Orthodox [ a ]           37   34      2     1   
2  78 in (198 cm) [ 1 ]        Orthodox           24   22      2   NaN   
3  76 in (193 cm) [ 1 ]        Orthodox           39   36      3   NaN   
4        80 in (203 cm)        Orthodox           26   26    NaN   NaN   

  No_contests  
0         NaN  
1         NaN  
2         NaN  
3         NaN  


In [12]:
df.fillna(0, inplace=True)
df

,Name,Age,Weight,Height,Reach,Stance,Total_fights,Wins,Losses,Draws,No_contests
0,Oleksandr Usyk,38,"Cruiserweight, Heavyweight",1.91 m (6 ft 3 in) [ 1 ],198 cm (78 in) [ 1 ],Southpaw,23,23,0,0,0
1,Tyson Fury,36,Heavyweight,6 ft 9 in (206 cm) [ 1 ],85 in (216 cm) [ 1 ],Orthodox [ a ],37,34,2,1,0
2,Daniel Dubois,27,Heavyweight,6 ft 5 in (196 cm) [ 1 ],78 in (198 cm) [ 1 ],Orthodox,24,22,2,0,0
3,Joseph Parker,33,Heavyweight,6 ft 4 in (193 cm) [ 1 ],76 in (193 cm) [ 1 ],Orthodox,39,36,3,0,0
4,Agit Kabayel,32,Heavyweight,6 ft 3 in (191 cm),80 in (203 cm),Orthodox,26,26,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
76,Masamichi Yabuki,32,"Light flyweight, Flyweight",5 ft 5 + 1 ⁄ 2 in (166 cm),64 + 1 ⁄ 2 in (164 cm),Orthodox,22,18,4,0,0
77,Elwin Soto,28,"Light flyweight, Flyweight",5 ft 3 in (160 cm),0,Orthodox,24,21,3,0,0
78,Shokichi Iwata,29,Light flyweight,5 ft 4 in (163 cm),64 in (163 cm),Orthodox,16,14,2,0,0
79,Oscar Collazo,0,0,0,0,0,0,0,0,0,0


In [13]:
### we need to create more variables in the dataset
import pandas as pd


import numpy as np
# Calculate KO Ratio as wins by KO divided by total wins
df['KO_Ratio'] = np.where(df['Wins'] > 0, df['Wins_by_KO'] / df['Wins'], 0)

# Optionally, format as percentage
df['KO_Ratio_percent'] = df['KO_Ratio'] * 100



### what if there is no winning streak

# Example: count KO wins from fight history if no direct KO column
df['Wins_by_KO'] = df['fight_history'].apply(
    lambda fights: sum(1 for fight in fights 
                       if fight.get('result') == 'W' and 'KO' in fight.get('method', ''))
)
df['KO_Ratio'] = np.where(df['Wins'] > 0, df['Wins_by_KO'] / df['Wins'], 0)


TypeError: '>' not supported between instances of 'str' and 'int'